In [14]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [15]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

stem = PorterStemmer()

[nltk_data] Downloading package stopwords to C:\Users\Purvi
[nltk_data]     jain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Importing Dataset

In [16]:
import pandas as pd 

df = pd.read_csv("fake_and_real_news.csv")

In [17]:
df.shape

(9900, 2)

In [18]:
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


# Text Cleaning

In [29]:
def clean_text(text):
    text = re.sub('[^a-zA-Z0-9]', ' ', str(text))
    text = text.lower()
    text = text.split()

    text = [
        stem.stem(word)
        for word in text
        if word not in stop_words
    ]

    return ' '.join(text)

In [30]:
df['clean_text'] = df['Text'].apply(clean_text)

In [31]:
df.head()

,Text,label,clean_text
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,top trump surrog brutal stab back pathet video...
1,U.S. conservative leader optimistic of common ...,Real,u conserv leader optimist common ground health...
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,trump propos u tax overhaul stir concern defic...
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,court forc ohio allow million illeg purg voter...
4,Democrats say Trump agrees to work on immigrat...,Real,democrat say trump agre work immigr bill wall ...


# Feature Extraction

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
x = vectorizer.fit_transform(df['clean_text'])
y = df['label']

# Training

- train_test_split randomly divides the data into training and testing sets.
- test_size=0.2 means 20% data is used for testing and 80% for training.
- random_state=42 ensures the same split happens every time you run the code, so results stay consistent.


In [38]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

# Model = Logistic Regression

In [39]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train, y_train)

LogisticRegression()

# Performance Metrics

In [41]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(x_test)

print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Classification Report \n", classification_report(y_test, y_pred))

Accuracy :  0.9949494949494949
Classification Report 
               precision    recall  f1-score   support

        Fake       1.00      0.99      0.99       973
        Real       0.99      1.00      1.00      1007

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980



In [42]:
y.value_counts()

label
Fake    5000
Real    4900
Name: count, dtype: int64

# Testing

In [49]:
test_news = [
    "Government announces free 10 lakh to every citizen starting next month"
]

test_clean = [clean_text(test_news[0])]
vector = vectorizer.transform(test_clean)
model.predict(vector)

array(['Real'], dtype=object)

In [50]:
test_news = [
"Scientists discover a miracle herb that cures cancer in 7 days"
]

test_clean = [clean_text(test_news[0])]
vector = vectorizer.transform(test_clean)
model.predict(vector)

array(['Fake'], dtype=object)

In [54]:
test_news = [
    "RBI increased the repo rate by 25 basis points to control inflation",
    "Supreme Court directed states to submit air pollution reports",
    "New study reveals that switching off mobile phones at night can increase human lifespan by 20 years.",
    "Scientists claim drinking hot water with lemon can completely cure COVID-19 and all future viruses within 48 hours."
]

test_clean = [clean_text(news) for news in test_news]
vector = vectorizer.transform(test_clean)
model.predict(vector)

array(['Real', 'Real', 'Fake', 'Fake'], dtype=object)

# Saving the Trained Model

In [ ]:
import pickle

pickle.dump(model, open("fake_news_model.pkl", "wb"))
pickle.dump(vectorizer)